In [1]:
import requests

url = "https://huggingface.co/datasets/printblue/EmoArt-130k/resolve/main/Annotation.json"
r = requests.get(url)

with open("Annotation.json", "wb") as f:
    f.write(r.content)

print("Downloaded Annotation.json")

Downloaded Annotation.json


In [2]:
import json

# Load the annotations
with open("Annotation.json", "r", encoding="utf-8") as f:
    annotations = json.load(f)

# Check the keys of the first item to understand structure
# Assuming the list contains dictionaries, access the first dictionary directly
first_item = annotations[0]
print(first_item.keys())

dict_keys(['request_id', 'description', 'image_path'])


In [3]:
from datasets import load_dataset

# Load as tar files, then decode both jpg + json
ds = load_dataset(
    "printblue/EmoArt-130k",
    split="train",
    streaming=True,
    trust_remote_code=True  # sometimes needed for custom decoding
)

for row in ds.take(5):
    print(row.keys())  # hoping to see 'json'

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'printblue/EmoArt-130k' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'printblue/EmoArt-130k' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


README.md: 0.00B [00:00, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


Resolving data files:   0%|          | 0/57 [00:00<?, ?it/s]

dict_keys(['jpg', '__key__', '__url__'])
dict_keys(['jpg', '__key__', '__url__'])
dict_keys(['jpg', '__key__', '__url__'])
dict_keys(['jpg', '__key__', '__url__'])
dict_keys(['jpg', '__key__', '__url__'])


In [4]:
from huggingface_hub import hf_hub_download
file = hf_hub_download("printblue/EmoArt-130k", "annotations.json")  # or csv

RepositoryNotFoundError: 404 Client Error. (Request ID: Root=1-6937e6f2-5b6e1d0d69b76c93582955b6;d3a1ca02-9821-4724-afa3-b7b463bad7a6)

Repository Not Found for url: https://huggingface.co/printblue/EmoArt-130k/resolve/main/annotations.json.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated. For more details, see https://huggingface.co/docs/huggingface_hub/authentication

In [5]:
def parse_record(example):
    meta = example.get("json", {})
    return {
        "description": meta.get("description", None),
        "dominant_emotion": meta.get("dominant_emotion", None),
        "style": meta.get("style", None),
        "region": meta.get("region", None),
        "image": example.get("jpg", None),
    }

parsed = ds.map(parse_record)

In [6]:
df = ds['train'].to_pandas()
print(df.shape)
print(df.columns.tolist())
df.head(3)

AttributeError: 'IterableColumn' object has no attribute 'to_pandas'